# Feature Engineering

This notebook converts the descriptive findings from the earlier monitoring phase into transaction-level features that can be used later in anomaly detection and supervised fraud modeling.

This is one of the most sensitive parts of the project because feature engineering can easily introduce leakage if it is handled carelessly. For that reason, the feature design in this notebook focuses on historical context only. Each feature is intended to represent what would have been knowable at or before the moment of the transaction, not what becomes visible afterward.

The goal of this phase is not to create as many features as possible. The goal is to create a compact, interpretable, and defensible feature layer that reflects transaction behavior, account history, balance context, and recent temporal activity.


## 1. Preparing the notebook environment

The first step configures the notebook so it can run reliably from the project root or from inside the `notebooks/` directory. It also sets dataframe display options so that wide feature tables remain readable during inspection.

This setup is intentionally simple, but it matters because the rest of the notebook depends on being able to locate the project folders and import the reusable feature-engineering functions from `src/`.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_OUTPUTS = PROJECT_ROOT / "data" / "outputs"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PROCESSED:", DATA_PROCESSED)
print("DATA_OUTPUTS:", DATA_OUTPUTS)

PROJECT_ROOT: /Users/twon/Documents/Time Series Analysis Project
DATA_PROCESSED: /Users/twon/Documents/Time Series Analysis Project/data/processed
DATA_OUTPUTS: /Users/twon/Documents/Time Series Analysis Project/data/outputs


In [3]:
print("Current working directory:", Path.cwd().resolve())
print("Project root exists:", PROJECT_ROOT.exists())
print("Processed folder exists:", DATA_PROCESSED.exists())

if DATA_PROCESSED.exists():
    print("Files in data/processed:")
    for p in sorted(DATA_PROCESSED.iterdir()):
        print("-", p.name)

Current working directory: /Users/twon/Documents/Time Series Analysis Project/notebooks
Project root exists: True
Processed folder exists: True
Files in data/processed:


## 2. Importing the reusable feature-engineering functions

This notebook relies on reusable source-code functions instead of writing all feature logic inline. That design keeps the notebook readable and makes the feature layer easier to test, reuse, and maintain.

The imported functions cover four broad areas:

- temporal helper columns
- sender-history features
- balance-based ratio features
- destination-history and sender-window features


In [4]:
from src.feature_engineering import (
    add_time_helpers,
    add_sender_history_features,
    add_balance_ratio_features,
    add_destination_history_features,
    add_sender_window_features,
    build_feature_inventory,
)

## 3. Locating the raw transaction dataset

Before any feature logic can run, the notebook needs to confirm which raw input file is available in `data/raw/`. The code checks a small list of candidate file names and selects the first existing match.

This is a practical project-hardening step. It prevents the notebook from silently depending on one fragile filename and makes the workflow easier to reuse across slightly different local setups.


In [5]:
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

candidate_raw_inputs = [
    RAW_DATA_DIR / "PS_20174392719_1491204439457_log.csv",
    RAW_DATA_DIR / "paysim dataset.csv",
    RAW_DATA_DIR / "transactions.csv",
]

existing_raw_inputs = [p for p in candidate_raw_inputs if p.exists()]
existing_raw_inputs

[PosixPath('/Users/twon/Documents/Time Series Analysis Project/data/raw/paysim dataset.csv')]

In [6]:
if not existing_raw_inputs:
    raise FileNotFoundError(
        "No raw transaction dataset found in data/raw/. "
        "Add the correct raw CSV filename to candidate_raw_inputs."
    )

INPUT_PATH = existing_raw_inputs[0]
print("Using raw input file:", INPUT_PATH.name)

Using raw input file: paysim dataset.csv


## 4. Loading and validating the raw transaction data

The notebook now loads the raw transaction file and confirms that the core columns required for feature construction are present.

This validation step is important because the rest of the notebook assumes that sender IDs, destination IDs, transaction amount, and balance columns are all available. If one of those columns were missing or malformed, the engineered features would either break or become analytically misleading.


In [7]:
df = pd.read_csv(INPUT_PATH).copy()

print(df.shape)
df.head()

(6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,"9,839.64",C1231006815,"170,136.00","160,296.36",M1979787155,0.00,0.00,0,0
1,1,PAYMENT,"1,864.28",C1666544295,"21,249.00","19,384.72",M2044282225,0.00,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,"21,182.00",0.00,1,0
4,1,PAYMENT,"11,668.14",C2048537720,"41,554.00","29,885.86",M1230701703,0.00,0.00,0,0


In [8]:
required_cols = [
    "step",
    "type",
    "amount",
    "nameOrig",
    "oldbalanceOrg",
    "newbalanceOrig",
    "nameDest",
]

missing_required_cols = [col for col in required_cols if col not in df.columns]
if missing_required_cols:
    raise ValueError(f"Missing required columns: {missing_required_cols}")

df = df.drop_duplicates().copy()
df["step"] = pd.to_numeric(df["step"], errors="coerce")
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df["oldbalanceOrg"] = pd.to_numeric(df["oldbalanceOrg"], errors="coerce")
df["newbalanceOrig"] = pd.to_numeric(df["newbalanceOrig"], errors="coerce")

df = df.dropna(subset=required_cols).reset_index(drop=True)

print(df.shape)

(6362620, 11)


## 5. Establishing transaction order

Time-aware feature engineering depends on a stable transaction order. This notebook creates a `row_id` column and sorts by `step` and `row_id` so that transactions occurring in the same hour still have a reproducible order.

That ordering decision is a small but meaningful safeguard. Without it, groupby-based historical features could behave inconsistently when multiple rows share the same `step` value.


In [9]:
df["row_id"] = np.arange(len(df))
df = df.sort_values(["step", "row_id"]).reset_index(drop=True)

print(df.shape)
df[["step", "row_id"]].head()

(6362620, 12)


,step,row_id
0,1,0
1,1,1
2,1,2
3,1,3
4,1,4


## 6. Creating the base temporal helper columns

These helper fields provide lightweight context used later in both analysis and modeling.

The `hour` column captures the hour-of-day position, while `day` groups the hourly steps into daily buckets aligned with the saved monitoring outputs from notebook `03`. The binary flags for transfer and cash-out activity are also useful because earlier analysis showed that fraud is concentrated in a small subset of transaction types.


In [10]:
df = add_time_helpers(df)

preview_cols = ["step", "hour", "day", "type", "is_transfer", "is_cash_out"]
existing_preview_cols = [col for col in preview_cols if col in df.columns]

df[existing_preview_cols].head()

,step,hour,day,type,is_transfer,is_cash_out
0,1,1,0,PAYMENT,0,0
1,1,1,0,PAYMENT,0,0
2,1,1,0,TRANSFER,1,0
3,1,1,0,CASH_OUT,0,1
4,1,1,0,PAYMENT,0,0


## 7. Building sender-history features

This step creates sender-side historical features using only prior activity from the same origin account.

These features answer questions such as:

- how many times has this sender transacted before now?
- how long has it been since the sender's prior transaction?
- how large was the previous transaction?
- what does the sender's historical average amount look like?

This is where the notebook begins turning raw rows into behavioral context.


In [11]:
df = add_sender_history_features(df)

sender_preview_cols = [
    "nameOrig",
    "step",
    "amount",
    "orig_txn_count_prior",
    "orig_amount_prior_sum",
    "orig_avg_amount_prior",
    "orig_prev_step",
    "orig_time_since_prev",
    "orig_prev_amount",
]

existing_sender_preview_cols = [col for col in sender_preview_cols if col in df.columns]
df[existing_sender_preview_cols].head(10)

,nameOrig,step,amount,orig_txn_count_prior,orig_amount_prior_sum,orig_avg_amount_prior,orig_prev_step,orig_time_since_prev,orig_prev_amount
0,C1231006815,1,"9,839.64",0,0.00,NaN,NaN,NaN,NaN
1,C1666544295,1,"1,864.28",0,0.00,NaN,NaN,NaN,NaN
2,C1305486145,1,181.00,0,0.00,NaN,NaN,NaN,NaN
3,C840083671,1,181.00,0,0.00,NaN,NaN,NaN,NaN
4,C2048537720,1,"11,668.14",0,0.00,NaN,NaN,NaN,NaN
5,C90045638,1,"7,817.71",0,0.00,NaN,NaN,NaN,NaN
6,C154988899,1,"7,107.77",0,0.00,NaN,NaN,NaN,NaN
7,C1912850431,1,"7,861.64",0,0.00,NaN,NaN,NaN,NaN
8,C1265012928,1,"4,024.36",0,0.00,NaN,NaN,NaN,NaN
9,C712410124,1,"5,337.77",0,0.00,NaN,NaN,NaN,NaN


## 8. Building balance-ratio features

The next block creates features that compare the transaction amount with the sender's pre-transaction balance and with the sender's expected balance change.

These features matter because fraud is often associated with draining behavior, unusually large balance shocks, or transactions that look extreme relative to the funds available before the transfer.


In [12]:
df = add_balance_ratio_features(df)

balance_preview_cols = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "orig_amount_to_oldbalance_ratio",
    "orig_balance_drop",
    "orig_balance_drop_ratio",
    "orig_amount_vs_prior_avg",
]

existing_balance_preview_cols = [col for col in balance_preview_cols if col in df.columns]
df[existing_balance_preview_cols].head(10)

,amount,oldbalanceOrg,newbalanceOrig,orig_amount_to_oldbalance_ratio,orig_balance_drop,orig_balance_drop_ratio,orig_amount_vs_prior_avg
0,"9,839.64","170,136.00","160,296.36",0.06,"9,839.64",0.06,NaN
1,"1,864.28","21,249.00","19,384.72",0.09,"1,864.28",0.09,NaN
2,181.00,181.00,0.00,1.00,181.00,1.00,NaN
3,181.00,181.00,0.00,1.00,181.00,1.00,NaN
4,"11,668.14","41,554.00","29,885.86",0.28,"11,668.14",0.28,NaN
5,"7,817.71","53,860.00","46,042.29",0.15,"7,817.71",0.15,NaN
6,"7,107.77","183,195.00","176,087.23",0.04,"7,107.77",0.04,NaN
7,"7,861.64","176,087.23","168,225.59",0.04,"7,861.64",0.04,NaN
8,"4,024.36","2,671.00",0.00,1.51,"2,671.00",1.00,NaN
9,"5,337.77","41,720.00","36,382.23",0.13,"5,337.77",0.13,NaN


## 9. Building destination-history features

Fraud risk is not only about the sender. Destination behavior can also matter, especially when funds are routed into newly active or unusually concentrated receiving accounts.

These features summarize what the destination account had previously received before the current transaction and how unusual the current amount looks relative to that prior history.


In [13]:
df = add_destination_history_features(df)

dest_preview_cols = [
    "nameDest",
    "step",
    "amount",
    "dest_txn_count_prior",
    "dest_amount_prior_sum",
    "dest_avg_amount_prior",
    "dest_prev_step",
    "dest_time_since_prev",
    "amount_vs_dest_prior_avg",
]

existing_dest_preview_cols = [col for col in dest_preview_cols if col in df.columns]

df.loc[
    df["dest_txn_count_prior"] > 0,
    existing_dest_preview_cols
].head(10)

,nameDest,step,amount,dest_txn_count_prior,dest_amount_prior_sum,dest_avg_amount_prior,dest_prev_step,dest_time_since_prev,amount_vs_dest_prior_avg
78,C1330106945,1,"42,712.39",1,"5,149.66","5,149.66",1.00,0.00,8.29
82,C766572210,1,"224,606.64",1,"136,872.92","136,872.92",1.00,0.00,1.64
86,C766572210,1,"554,026.99",2,"361,479.56","180,739.78",1.00,0.00,3.07
88,C1590550415,1,"761,507.39",1,"379,856.23","379,856.23",1.00,0.00,2.00
89,C1590550415,1,"1,429,051.47",2,"1,141,363.62","570,681.81",1.00,0.00,2.50
90,C392292416,1,"358,831.92",1,"125,872.53","125,872.53",1.00,0.00,2.85
91,C1359044626,1,"367,768.40",1,"147,543.10","147,543.10",1.00,0.00,2.49
92,C1509514333,1,"209,711.11",1,"110,414.71","110,414.71",1.00,0.00,1.90
94,C1590550415,1,"1,724,887.05",3,"2,570,415.09","856,805.03",1.00,0.00,2.01
95,C1359044626,1,"710,544.77",2,"515,311.50","257,655.75",1.00,0.00,2.76


## 10. Building sender rolling-window features

The final major feature block captures short-window sender activity over the prior 1, 6, and 24 steps.

These windows are especially useful for fraud work because they can expose burst behavior that is hard to see from a single transaction alone. A sender that suddenly creates several transfers in a short period may deserve more attention than one making the same amount as part of a long, steady pattern.


In [14]:
df = add_sender_window_features(df, windows=(1, 6, 24))

window_fill_cols = [
    "orig_txn_count_last_1",
    "orig_amount_sum_last_1",
    "orig_txn_count_last_6",
    "orig_amount_sum_last_6",
    "orig_txn_count_last_24",
    "orig_amount_sum_last_24",
]

existing_window_fill_cols = [col for col in window_fill_cols if col in df.columns]
df[existing_window_fill_cols] = df[existing_window_fill_cols].fillna(0)

window_preview_cols = [
    "nameOrig",
    "step",
    "orig_txn_count_prior",
    "orig_txn_count_last_1",
    "orig_amount_sum_last_1",
    "orig_txn_count_last_6",
    "orig_amount_sum_last_6",
    "orig_txn_count_last_24",
    "orig_amount_sum_last_24",
]

existing_window_preview_cols = [col for col in window_preview_cols if col in df.columns]

window_preview = df.loc[
    df["orig_txn_count_prior"] > 0,
    existing_window_preview_cols
].head(10)

print(df[existing_window_fill_cols].isna().sum())
window_preview

orig_txn_count_last_1      0
orig_amount_sum_last_1     0
orig_txn_count_last_6      0
orig_amount_sum_last_6     0
orig_txn_count_last_24     0
orig_amount_sum_last_24    0
dtype: int64


,nameOrig,step,orig_txn_count_prior,orig_txn_count_last_1,orig_amount_sum_last_1,orig_txn_count_last_6,orig_amount_sum_last_6,orig_txn_count_last_24,orig_amount_sum_last_24
115385,C1709295811,11,1,1.00,"497,725.93",1.00,"497,725.93",1.00,"497,725.93"
146871,C44568807,12,1,1.00,"6,107.51",1.00,"6,107.51",1.00,"6,107.51"
148517,C260230637,12,1,1.00,"12,712.93",1.00,"12,712.93",1.00,"12,712.93"
196159,C745009740,13,1,1.00,"41,726.55",1.00,"41,726.55",1.00,"41,726.55"
208603,C1842781381,13,1,1.00,"4,231.32",1.00,"4,231.32",1.00,"4,231.32"
208779,C199116739,13,1,1.00,"11,813.50",1.00,"11,813.50",1.00,"11,813.50"
235870,C779875094,14,1,1.00,"31,395.75",1.00,"31,395.75",1.00,"31,395.75"
236407,C189326840,14,1,1.00,"130,477.15",1.00,"130,477.15",1.00,"130,477.15"
272911,C1250194175,15,1,1.00,"452,480.26",1.00,"452,480.26",1.00,"452,480.26"
281403,C746558292,15,1,1.00,"149,585.24",1.00,"149,585.24",1.00,"149,585.24"


In [15]:
df.loc[df["orig_txn_count_prior"] > 0, [
    "nameOrig",
    "step",
    "orig_txn_count_prior",
    "orig_txn_count_last_1",
    "orig_amount_sum_last_1",
    "orig_txn_count_last_6",
    "orig_amount_sum_last_6",
    "orig_txn_count_last_24",
    "orig_amount_sum_last_24",
]].head(10)

,nameOrig,step,orig_txn_count_prior,orig_txn_count_last_1,orig_amount_sum_last_1,orig_txn_count_last_6,orig_amount_sum_last_6,orig_txn_count_last_24,orig_amount_sum_last_24
115385,C1709295811,11,1,1.00,"497,725.93",1.00,"497,725.93",1.00,"497,725.93"
146871,C44568807,12,1,1.00,"6,107.51",1.00,"6,107.51",1.00,"6,107.51"
148517,C260230637,12,1,1.00,"12,712.93",1.00,"12,712.93",1.00,"12,712.93"
196159,C745009740,13,1,1.00,"41,726.55",1.00,"41,726.55",1.00,"41,726.55"
208603,C1842781381,13,1,1.00,"4,231.32",1.00,"4,231.32",1.00,"4,231.32"
208779,C199116739,13,1,1.00,"11,813.50",1.00,"11,813.50",1.00,"11,813.50"
235870,C779875094,14,1,1.00,"31,395.75",1.00,"31,395.75",1.00,"31,395.75"
236407,C189326840,14,1,1.00,"130,477.15",1.00,"130,477.15",1.00,"130,477.15"
272911,C1250194175,15,1,1.00,"452,480.26",1.00,"452,480.26",1.00,"452,480.26"
281403,C746558292,15,1,1.00,"149,585.24",1.00,"149,585.24",1.00,"149,585.24"


## 11. Bringing in the earlier monitoring outputs for context

Feature engineering should not happen in isolation from the earlier exploratory findings. This step pulls in saved monitoring outputs from notebook `03` so the feature phase can stay grounded in what the descriptive analysis already showed.

This helps connect the project phases logically: the monitoring notebook identified risky periods and transaction patterns, and the feature notebook now considers how that insight can be translated into row-level predictors.


In [34]:
PHASE_3_OUTPUTS = DATA_OUTPUTS / "phase_3_eda_monitoring"

monitoring_hour_path = PHASE_3_OUTPUTS / "hourly_monitoring.csv"
monitoring_day_path = PHASE_3_OUTPUTS / "daily_monitoring.csv"

print("hourly_monitoring exists:", monitoring_hour_path.exists())
print("daily_monitoring exists:", monitoring_day_path.exists())

hourly_monitoring exists: True
daily_monitoring exists: True


In [35]:
if monitoring_hour_path.exists():
    hourly_monitoring = pd.read_csv(monitoring_hour_path)
    print("hourly_monitoring shape:", hourly_monitoring.shape)
    display(hourly_monitoring.head())
else:
    hourly_monitoring = None

if monitoring_day_path.exists():
    daily_monitoring = pd.read_csv(monitoring_day_path)
    print("daily_monitoring shape:", daily_monitoring.shape)
    display(daily_monitoring.head())
else:
    daily_monitoring = None

hourly_monitoring shape: (743, 16)


,hour,txn_count,fraud_count,total_amount,avg_amount,fraud_amount,fraud_rate,fraud_amount_share,txn_count_roll_mean_24,fraud_count_roll_mean_24,fraud_rate_roll_mean_24,fraud_rate_roll_std_24,avg_amount_roll_mean_24,fraud_rate_zscore_like,fraud_count_spike_flag,fraud_rate_spike_flag
0,1,2708,16,"285,429,200.00","105,402.21","3,740,247.00",0.01,0.01,"2,708.00",16.00,0.01,0.00,"105,402.21",0.00,False,False
1,2,1014,8,"85,921,610.00","84,735.31","4,186,592.50",0.01,0.05,"1,861.00",12.00,0.01,0.00,"95,068.76",0.71,False,False
2,3,552,4,"43,293,884.00","78,430.95","66,832.74",0.01,0.00,"1,424.67",9.33,0.01,0.00,"89,522.83",0.23,False,False
3,4,565,10,"72,910,030.00","129,044.30","26,400,274.00",0.02,0.36,"1,209.75",9.50,0.01,0.01,"99,403.20",1.48,False,False
4,5,665,6,"45,548,090.00","68,493.37","381,841.53",0.01,0.01,"1,100.80",8.80,0.01,0.00,"93,221.23",-0.11,False,False


daily_monitoring shape: (31, 14)


,day,txn_count,fraud_count,total_amount,avg_amount,fraud_amount,fraud_rate,fraud_amount_share,txn_count_roll_mean_7,fraud_count_roll_mean_7,fraud_rate_roll_mean_7,fraud_rate_roll_std_7,fraud_rate_zscore_like,fraud_rate_spike_flag
0,0,574255,271,"92,131,870,000.00","160,437.20","211,163,820.00",0.00,0.00,"574,255.00",271.00,0.00,0.00,0.00,False
1,1,455238,309,"71,238,640,000.00","156,486.58","379,239,680.00",0.00,0.01,"514,746.50",290.00,0.00,0.00,0.71,False
2,2,1070,310,"447,084,480.00","417,835.97","394,508,860.00",0.29,0.88,"343,521.00",296.67,0.10,0.17,1.15,False
3,3,28240,262,"3,830,547,500.00","135,642.61","438,246,850.00",0.01,0.11,"264,700.75",288.00,0.08,0.14,-0.46,False
4,4,9789,252,"1,325,133,800.00","135,369.69","244,980,590.00",0.03,0.18,"213,718.40",280.80,0.07,0.13,-0.31,False


In [36]:
if high_risk_hours is not None:
    df = df.merge(high_risk_hours, on="hour", how="left")

if high_risk_days is not None:
    df = df.merge(high_risk_days, on="day", how="left")

print(df.shape)

(6362620, 38)


## 12. Reviewing risk and spike-oriented columns

This quick inspection checks whether any risk-like or spike-related fields are already present after the earlier transformations and imported context.

The goal is not to keep every possible field. The goal is to understand what signals are available and to avoid carrying forward columns that would be redundant, weakly defined, or potentially leakage-prone.


In [37]:
flag_cols = [col for col in df.columns if ("risk" in col.lower()) or ("spike" in col.lower())]

for col in flag_cols:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(0)

flag_cols

[]

## 13. Defining the final feature set for downstream modeling

This is the key feature-selection step. The notebook explicitly separates columns that should be dropped from the final modeling feature set and then defines the core engineered features that are intended to survive into downstream anomaly detection and supervised modeling.

Being explicit here is important. A tight feature list is easier to defend than a large, unclear one, and it reduces the risk of quietly carrying forward identifiers or columns that would not belong in a realistic model pipeline.


In [38]:
drop_cols = [
    "orig_amount_cum",
    "dest_amount_cum",
]

drop_cols_existing = [col for col in drop_cols if col in df.columns]
df = df.drop(columns=drop_cols_existing)

drop_cols_existing

[]

In [39]:
core_feature_cols = [
    "orig_txn_count_prior",
    "orig_amount_prior_sum",
    "orig_avg_amount_prior",
    "orig_prev_step",
    "orig_time_since_prev",
    "orig_prev_amount",
    "orig_amount_to_oldbalance_ratio",
    "orig_balance_drop",
    "orig_balance_drop_ratio",
    "orig_amount_vs_prior_avg",
    "dest_txn_count_prior",
    "dest_amount_prior_sum",
    "dest_avg_amount_prior",
    "dest_prev_step",
    "dest_time_since_prev",
    "amount_vs_dest_prior_avg",
    "orig_txn_count_last_1",
    "orig_amount_sum_last_1",
    "orig_txn_count_last_6",
    "orig_amount_sum_last_6",
    "orig_txn_count_last_24",
    "orig_amount_sum_last_24",
]

feature_cols = [col for col in core_feature_cols if col in df.columns]
feature_cols

['orig_txn_count_prior',
 'orig_amount_prior_sum',
 'orig_avg_amount_prior',
 'orig_prev_step',
 'orig_time_since_prev',
 'orig_prev_amount',
 'orig_amount_to_oldbalance_ratio',
 'orig_balance_drop',
 'orig_balance_drop_ratio',
 'orig_amount_vs_prior_avg',
 'dest_txn_count_prior',
 'dest_amount_prior_sum',
 'dest_avg_amount_prior',
 'dest_prev_step',
 'dest_time_since_prev',
 'amount_vs_dest_prior_avg',
 'orig_txn_count_last_1',
 'orig_amount_sum_last_1',
 'orig_txn_count_last_6',
 'orig_amount_sum_last_6',
 'orig_txn_count_last_24',
 'orig_amount_sum_last_24']

## 14. Auditing null and infinite values in the engineered feature set

Engineered features often introduce nulls or infinite values, especially when ratios depend on prior counts or prior balances. This section checks the final feature set carefully before export.

That review is essential because a feature can be conceptually strong and still be operationally awkward if it creates large amounts of undefined or unstable values.


In [40]:
null_summary = (
    df[feature_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("null_rate")
    .reset_index()
    .rename(columns={"index": "feature"})
)

null_summary

,feature,null_rate
0,orig_avg_amount_prior,1.00
1,orig_prev_step,1.00
2,orig_time_since_prev,1.00
3,orig_prev_amount,1.00
4,orig_amount_vs_prior_avg,1.00
5,amount_vs_dest_prior_avg,0.43
6,dest_avg_amount_prior,0.43
7,dest_prev_step,0.43
8,dest_time_since_prev,0.43
9,orig_amount_to_oldbalance_ratio,0.33


In [49]:
inf_issues = inf_summary[inf_summary["inf_count"] > 0]

if inf_issues.empty:
    print("No infinite values found in numeric feature columns.")
else:
    display(inf_issues)

No infinite values found in numeric feature columns.


## 15. Reviewing the feature distributions

This descriptive summary provides a final sanity check on the engineered numeric fields.

The point is not only to confirm that the code ran. It is also to see whether the feature ranges and medians are plausible, whether they are dominated by edge cases, and whether the variables appear informative enough to justify carrying them into the later modeling phases.


In [42]:
df[feature_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
orig_txn_count_prior,"6,362,620.00",0.00,0.04,0.00,0.00,0.00,0.00,2.00
orig_amount_prior_sum,"6,362,620.00",260.71,"29,797.01",0.00,0.00,0.00,0.00,"46,211,920.92"
orig_avg_amount_prior,"9,313.00","177,898.99","758,233.71",0.52,"12,664.96","71,604.78","208,727.72","46,211,920.92"
orig_prev_step,"9,313.00",162.08,110.76,1.00,41.00,163.00,237.00,688.00
orig_time_since_prev,"9,313.00",158.31,123.66,0.00,56.00,139.00,225.00,704.00
orig_prev_amount,"9,313.00","177,900.60","758,239.04",0.52,"12,664.96","71,516.85","208,902.74","46,211,920.92"
orig_amount_to_oldbalance_ratio,"4,260,171.00",122.04,"4,149.28",0.00,0.07,0.74,6.66,"3,925,475.60"
orig_balance_drop,"6,362,620.00","-21,230.56","146,643.29","-1,915,267.90",0.00,0.00,"10,150.44","10,000,000.00"
orig_balance_drop_ratio,"4,260,171.00",-23.74,"1,095.13","-548,909.81",-0.02,0.20,1.00,1.00
orig_amount_vs_prior_avg,"9,313.00",95.70,"4,228.09",0.00,0.18,1.06,5.88,"316,843.14"


In [43]:
sample_cols = [
    "step",
    "type",
    "amount",
    "nameOrig",
    "nameDest",
] + feature_cols

sample_cols_existing = [col for col in sample_cols if col in df.columns]
df[sample_cols_existing].head(10)

,step,type,amount,nameOrig,nameDest,orig_txn_count_prior,orig_amount_prior_sum,orig_avg_amount_prior,orig_prev_step,orig_time_since_prev,orig_prev_amount,orig_amount_to_oldbalance_ratio,orig_balance_drop,orig_balance_drop_ratio,orig_amount_vs_prior_avg,dest_txn_count_prior,dest_amount_prior_sum,dest_avg_amount_prior,dest_prev_step,dest_time_since_prev,amount_vs_dest_prior_avg,orig_txn_count_last_1,orig_amount_sum_last_1,orig_txn_count_last_6,orig_amount_sum_last_6,orig_txn_count_last_24,orig_amount_sum_last_24
0,1,PAYMENT,"9,839.64",C1231006815,M1979787155,0,0.00,NaN,NaN,NaN,NaN,0.06,"9,839.64",0.06,NaN,0,0.00,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00
1,1,PAYMENT,"1,864.28",C1666544295,M2044282225,0,0.00,NaN,NaN,NaN,NaN,0.09,"1,864.28",0.09,NaN,0,0.00,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00
2,1,TRANSFER,181.00,C1305486145,C553264065,0,0.00,NaN,NaN,NaN,NaN,1.00,181.00,1.00,NaN,0,0.00,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00
3,1,CASH_OUT,181.00,C840083671,C38997010,0,0.00,NaN,NaN,NaN,NaN,1.00,181.00,1.00,NaN,0,0.00,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00
4,1,PAYMENT,"11,668.14",C2048537720,M1230701703,0,0.00,NaN,NaN,NaN,NaN,0.28,"11,668.14",0.28,NaN,0,0.00,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00
5,1,PAYMENT,"7,817.71",C90045638,M573487274,0,0.00,NaN,NaN,NaN,NaN,0.15,"7,817.71",0.15,NaN,0,0.00,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00
6,1,PAYMENT,"7,107.77",C154988899,M408069119,0,0.00,NaN,NaN,NaN,NaN,0.04,"7,107.77",0.04,NaN,0,0.00,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00
7,1,PAYMENT,"7,861.64",C1912850431,M633326333,0,0.00,NaN,NaN,NaN,NaN,0.04,"7,861.64",0.04,NaN,0,0.00,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00
8,1,PAYMENT,"4,024.36",C1265012928,M1176932104,0,0.00,NaN,NaN,NaN,NaN,1.51,"2,671.00",1.00,NaN,0,0.00,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00
9,1,DEBIT,"5,337.77",C712410124,C195600860,0,0.00,NaN,NaN,NaN,NaN,0.13,"5,337.77",0.13,NaN,0,0.00,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00


## 16. Building the feature inventory

A polished project should not leave features undocumented. This step creates a feature inventory that explains what each retained feature measures and why it might matter for fraud risk.

That inventory becomes part of the technical handoff because it helps future readers understand the analytical meaning of the dataset rather than treating the engineered columns as opaque model inputs.


In [44]:
feature_inventory = build_feature_inventory()
feature_inventory.head(20)

,feature_name,what_it_measures,why_it_might_indicate_risk,strictly_past_only,safe_for_modeling
0,orig_txn_count_prior,Number of prior transactions by sender,High prior activity may indicate burst behavio...,True,True
1,orig_time_since_prev,Time gap since sender's previous transaction,Very short gaps may indicate rapid transaction...,True,True
2,orig_amount_vs_prior_avg,Current amount relative to sender's prior aver...,Large deviations from normal transaction size ...,True,True
3,amount_vs_dest_prior_avg,Current amount relative to receiver's prior av...,Sudden jumps in receiving behavior may indicat...,True,True


## 17. Saving the engineered artifacts

The feature dataset and the feature inventory are now saved to disk so that later notebooks can use them without rerunning the full feature-engineering workflow.

Persisting both artifacts is important. The parquet file supports efficient downstream analysis, while the feature inventory preserves interpretability and project documentation.


In [45]:
FEATURES_PATH = DATA_PROCESSED / "model_features.parquet"
FEATURE_INVENTORY_PATH = DATA_OUTPUTS / "feature_inventory.csv"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_OUTPUTS.mkdir(parents=True, exist_ok=True)

In [46]:
df.to_parquet(FEATURES_PATH, index=False)
feature_inventory.to_csv(FEATURE_INVENTORY_PATH, index=False)

print("Saved feature dataset to:", FEATURES_PATH)
print("Saved feature inventory to:", FEATURE_INVENTORY_PATH)

Saved feature dataset to: /Users/twon/Documents/Time Series Analysis Project/data/processed/model_features.parquet
Saved feature inventory to: /Users/twon/Documents/Time Series Analysis Project/data/outputs/feature_inventory.csv


## 18. Final quality check and phase closeout

The last cells confirm the final dataset shape and show the retained feature list. This acts as a final checkpoint before the project moves into anomaly detection and supervised modeling.

At this point, the project has moved from descriptive monitoring into a reusable transaction-level feature layer that is substantially richer than the raw data and still anchored to historically available information.


In [47]:
print(df.shape)
df.head()

(6362620, 38)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,row_id,hour,day,is_transfer,is_cash_out,orig_txn_count_prior,orig_amount_prior_sum,orig_avg_amount_prior,orig_prev_step,orig_time_since_prev,orig_prev_amount,orig_amount_to_oldbalance_ratio,orig_balance_drop,orig_balance_drop_ratio,orig_amount_vs_prior_avg,dest_txn_count_prior,dest_amount_prior_sum,dest_avg_amount_prior,dest_prev_step,dest_time_since_prev,amount_vs_dest_prior_avg,orig_txn_count_last_1,orig_amount_sum_last_1,orig_txn_count_last_6,orig_amount_sum_last_6,orig_txn_count_last_24,orig_amount_sum_last_24
0,1,PAYMENT,"9,839.64",C1231006815,"170,136.00","160,296.36",M1979787155,0.00,0.00,0,0,0,1,0,0,0,0,0.00,NaN,NaN,NaN,NaN,0.06,"9,839.64",0.06,NaN,0,0.00,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00
1,1,PAYMENT,"1,864.28",C1666544295,"21,249.00","19,384.72",M2044282225,0.00,0.00,0,0,1,1,0,0,0,0,0.00,NaN,NaN,NaN,NaN,0.09,"1,864.28",0.09,NaN,0,0.00,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0,2,1,0,1,0,0,0.00,NaN,NaN,NaN,NaN,1.00,181.00,1.00,NaN,0,0.00,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,"21,182.00",0.00,1,0,3,1,0,0,1,0,0.00,NaN,NaN,NaN,NaN,1.00,181.00,1.00,NaN,0,0.00,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00
4,1,PAYMENT,"11,668.14",C2048537720,"41,554.00","29,885.86",M1230701703,0.00,0.00,0,0,4,1,0,0,0,0,0.00,NaN,NaN,NaN,NaN,0.28,"11,668.14",0.28,NaN,0,0.00,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00


In [48]:
sorted(feature_cols)

['amount_vs_dest_prior_avg',
 'dest_amount_prior_sum',
 'dest_avg_amount_prior',
 'dest_prev_step',
 'dest_time_since_prev',
 'dest_txn_count_prior',
 'orig_amount_prior_sum',
 'orig_amount_sum_last_1',
 'orig_amount_sum_last_24',
 'orig_amount_sum_last_6',
 'orig_amount_to_oldbalance_ratio',
 'orig_amount_vs_prior_avg',
 'orig_avg_amount_prior',
 'orig_balance_drop',
 'orig_balance_drop_ratio',
 'orig_prev_amount',
 'orig_prev_step',
 'orig_time_since_prev',
 'orig_txn_count_last_1',
 'orig_txn_count_last_24',
 'orig_txn_count_last_6',
 'orig_txn_count_prior']